# 🧠 Model Training and Evaluation
The objective is to compare multiple regression algorithms, identify the best-performing model, optimize its performance through hyperparameter tuning, and prepare it for deployment in the prediction pipeline.

In [22]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from src.config import TRAIN_PROCESSED_PATH
from src.config import MODEL_PATH
from src.config import FEATURE_PATH
from src.feature_engineering import evaluate_model

import pandas as pd
import numpy as np
from scipy.stats import randint, uniform

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [ ]:
## Load Feature Engineered Dataset

df = pd.read_csv(TRAIN_PROCESSED_PATH)

In [16]:
print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (1447260, 23)


,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_year,pickup_month,pickup_day,...,is_weekend,dropoff_year,dropoff_month,dropoff_day,dropoff_weekday,dropoff_hour,dropoff_minute,haversine_distance,manhattan_distance,bearing
0,1,-73.982155,40.767937,-73.964630,40.765602,0,455,2016,3,14,...,0,2016,3,14,0,17,32,1.498521,0.019859,99.970196
1,1,-73.980415,40.738564,-73.999481,40.731152,0,663,2016,6,12,...,1,2016,6,12,6,0,54,1.805507,0.026478,242.846232
2,1,-73.979027,40.763939,-74.005333,40.710087,0,2124,2016,1,19,...,0,2016,1,19,1,12,10,6.385098,0.080158,200.319835
3,1,-74.010040,40.719971,-74.012268,40.706718,0,429,2016,4,6,...,0,2016,4,6,2,19,39,1.485498,0.015480,187.262300
4,1,-73.973053,40.793209,-73.972923,40.782520,0,435,2016,3,26,...,1,2016,3,26,5,13,38,1.188588,0.010818,179.473585


In [ ]:
## Separate Features and Target Variable

X = df.drop('trip_duration', axis=1)
y = np.log1p(df['trip_duration'])

In [17]:
print(f"Features Shape : {X.shape}")
print(f"Target Shape   : {y.shape}")

Features Shape : (1447260, 22)
Target Shape   : (1447260,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")
print(f"Number of Features : {X_train.shape[1]}")

Training Samples : 1157808
Testing Samples  : 289452
Number of Features : 22


## Linear Regression Model

In [8]:
# Initialize the model
lr_model = LinearRegression()

# Train the model
lr_model.fit(X_train, y_train)

LinearRegression()

In [9]:
# Predict on test data
y_pred_lr = lr_model.predict(X_test)

In [11]:
mae_lr, mse_lr, rmse_lr, r2_lr = evaluate_model(
    "Linear Regression",
    y_test,
    y_pred_lr
)


Linear Regression
----------------------------------------
MAE  : 0.37
MSE  : 0.25
RMSE : 0.50
R²   : 0.5205


## Decision Tree Regressor

In [ ]:
# Initialize model
dt_model = DecisionTreeRegressor(
    random_state=42
)

# Train model
dt_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [15]:
y_pred_dt = dt_model.predict(X_test)

In [16]:
mae_dt, mse_dt, rmse_dt, r2_dt = evaluate_model(
    "Decision Tree Regression",
    y_test,
    y_pred_dt
)

Linear Regression Performance
----------------------------------------
MAE  : 0.3186233829981255
MSE  : 0.19124722211299042
RMSE : 0.4373182160772524
R²   : 0.6387486157183537


## Random Forest Regressor

In [18]:
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2, 
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Train model
rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=15, max_features='sqrt', min_samples_leaf=2,
                      min_samples_split=5, n_estimators=50, n_jobs=-1,
                      random_state=42)

In [19]:
y_pred_rf = rf.predict(X_test)

In [20]:
mae_rf, mse_lr, rmse_rf, r2_rf = evaluate_model(
    "Random Forest Regression",
    y_test,
    y_pred_rf
)

Linear Regression Performance
----------------------------------------
MAE  : 0.25390437601703725
MSE  : 0.11701208469406436
RMSE : 0.34207029203668704
R²   : 0.7789731160202784


## Gradient Boosting Regressor

In [22]:
# Initialize model
gb_model = GradientBoostingRegressor(
    n_estimators=30,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2, 
    max_features="sqrt",
    learning_rate=0.1,
    random_state=42
)

# Train model
gb_model.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=15, max_features='sqrt', min_samples_leaf=2,
                          min_samples_split=5, n_estimators=30,
                          random_state=42)

In [23]:
y_pred_gb = gb_model.predict(X_test)

In [24]:
mae_gb, mse_gb, rmse_gb, r2_gb = evaluate_model(
    "XGBoost Regression",
    y_test,
    y_pred_gb
)

Linear Regression Performance
----------------------------------------
MAE  : 0.22208200445044113
MSE  : 0.09147943598544521
RMSE : 0.30245567606749457
R²   : 0.8272023378016866


## Model Performance Comparison

Compare the performance of all trained regression models using MAE, MSE, RMSE, and R² Score.

This comparison identifies the strongest baseline model before performing hyperparameter optimization.

In [27]:
comparison_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        mae_lr,
        mae_dt,
        mae_rf,
        mae_gb
    ],
    "MSE": [
        mse_lr,
        mse_dt,
        mse_rf,
        mse_gb
    ],
    "RMSE": [
        rmse_lr,
        rmse_dt,
        rmse_rf,
        rmse_gb

    ],
    "R² Score": [
        r2_lr,
        r2_dt,
        r2_rf,
        r2_gb
    ]
})

# Sort by best R² Score
comparison_df = comparison_df.sort_values(
    by="R² Score",
    ascending=False
).reset_index(drop=True)

# Round values for better readability
comparison_df = comparison_df.round({
    "MAE": 2,
    "MSE": 2,
    "RMSE": 2,
    "R² Score": 4
})

comparison_df

,Model,MAE,MSE,RMSE,R² Score
0,Gradient Boosting,0.22,0.09,0.30,0.8272
1,Random Forest,0.25,0.12,0.34,0.7790
2,Decision Tree,0.32,0.19,0.44,0.6387
3,Linear Regression,0.37,0.25,0.50,0.5205


## Hyperparameter Tuning using RandomizedSearchCV

Optimize the XGBoost model by searching different combinations of hyperparameters using RandomizedSearchCV.

In [13]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Hyperparameter search space
param_dist = {
    "n_estimators": randint(200, 1000),
    "learning_rate": uniform(0.01, 0.29),
    "max_depth": randint(3, 10),
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "gamma": uniform(0, 5),
    "reg_alpha": uniform(0, 1),
    "reg_lambda": uniform(0, 2)
}

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=5,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [14]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                        'reg_alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001819AF650F0>,
                                        'reg_lambda': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001819AEFBD10>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001819AF80CD0>},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)

In [15]:
print("Best Parameters:")
print(random_search.best_params_)
print("Best CV RMSE:", -random_search.best_score_)
best_model = random_search.best_estimator_

Best Parameters:
{'colsample_bytree': np.float64(0.7334834444556088), 'gamma': np.float64(0.7143340896097039), 'learning_rate': np.float64(0.19875765715516733), 'max_depth': 7, 'min_child_weight': 2, 'n_estimators': 543, 'reg_alpha': np.float64(0.8324426408004217), 'reg_lambda': np.float64(0.4246782213565523), 'subsample': np.float64(0.6727299868828402)}
Best CV RMSE: 0.1205143967434013


In [16]:
y_pred_xgb = best_model.predict(X_test)

In [23]:
mae_xgb, mse_xgb, rmse_xgb, r2_xgb = evaluate_model(
    "XGBoost Regression with hyperparameter tuning",
    y_test,
    y_pred_xgb
)


XGBoost Regression with hyperparameter tuning
----------------------------------------
MAE  : 0.08
MSE  : 0.01
RMSE : 0.11
R²   : 0.9765


- Hyperparameter tuning improves the model's predictive performance.

## Save the Final Model

Save the optimized XGBoost model and the feature column order using Joblib.

In [14]:
# Save the trained model
joblib.dump(best_model,MODEL_PATH)

# Save the feature column order
joblib.dump(X_train.columns.tolist(), FEATURE_PATH)

print("✅ Model saved successfully!")
print("📁 xgboost_model.pkl")
print("📁 feature_columns.pkl")

✅ Model saved successfully!
📁 xgboost_model.pkl
📁 feature_columns.pkl


## Conclusion

In this notebook, multiple regression algorithms were trained and evaluated to predict NYC taxi trip duration. After comparing the baseline models, XGBoost was selected and optimized using RandomizedSearchCV to achieve the best predictive performance.